# Learning Cross Validation Techiniques

In [1]:
# import necessary Libariers

import numpy as np
from sklearn.datasets import load_iris

from sklearn.model_selection import KFold, cross_val_score

from sklearn.linear_model import LogisticRegression

In [2]:
iris = load_iris()
X, y = iris.data, iris.target
X[:5]

array([[5.1, 3.5, 1.4, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [4.7, 3.2, 1.3, 0.2],
       [4.6, 3.1, 1.5, 0.2],
       [5. , 3.6, 1.4, 0.2]])

In [3]:
y[:5]

array([0, 0, 0, 0, 0])

In [4]:
model = LogisticRegression(max_iter = 200, n_jobs= None)

cv = KFold(n_splits =5, shuffle = True, random_state =42)

scores = cross_val_score(model, X,y, cv=cv,scoring = "accuracy")

In [5]:
print("Fold Accuracy :", scores)
print("Mean Accuracy:", scores.mean())
print("Std dev", scores.std())
print("Approx 95% CI: [{:.3f}, {:.3f}]".format(scores.mean() -1.96*scores.std()/np.sqrt(len(scores)),
                                               scores.mean() +1.96*scores.std()/np.sqrt(len(scores))))

Fold Accuracy : [1.         1.         0.93333333 0.96666667 0.96666667]
Mean Accuracy: 0.9733333333333334
Std dev 0.024944382578492935
Approx 95% CI: [0.951, 0.995]


# 10 Fold stratified Cross Validation

## StratifiedKFlod+ Multiple Metrics+Pipeline

### Dataset Breast Cancer

In [6]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import confusion_matrix,classification_report, roc_auc_score

In [7]:
data = load_breast_cancer()
X,y = data.data, data.target
X[:5]

array([[1.799e+01, 1.038e+01, 1.228e+02, 1.001e+03, 1.184e-01, 2.776e-01,
        3.001e-01, 1.471e-01, 2.419e-01, 7.871e-02, 1.095e+00, 9.053e-01,
        8.589e+00, 1.534e+02, 6.399e-03, 4.904e-02, 5.373e-02, 1.587e-02,
        3.003e-02, 6.193e-03, 2.538e+01, 1.733e+01, 1.846e+02, 2.019e+03,
        1.622e-01, 6.656e-01, 7.119e-01, 2.654e-01, 4.601e-01, 1.189e-01],
       [2.057e+01, 1.777e+01, 1.329e+02, 1.326e+03, 8.474e-02, 7.864e-02,
        8.690e-02, 7.017e-02, 1.812e-01, 5.667e-02, 5.435e-01, 7.339e-01,
        3.398e+00, 7.408e+01, 5.225e-03, 1.308e-02, 1.860e-02, 1.340e-02,
        1.389e-02, 3.532e-03, 2.499e+01, 2.341e+01, 1.588e+02, 1.956e+03,
        1.238e-01, 1.866e-01, 2.416e-01, 1.860e-01, 2.750e-01, 8.902e-02],
       [1.969e+01, 2.125e+01, 1.300e+02, 1.203e+03, 1.096e-01, 1.599e-01,
        1.974e-01, 1.279e-01, 2.069e-01, 5.999e-02, 7.456e-01, 7.869e-01,
        4.585e+00, 9.403e+01, 6.150e-03, 4.006e-02, 3.832e-02, 2.058e-02,
        2.250e-02, 4.571e-03, 2.357e

In [8]:
y[:5]

array([0, 0, 0, 0, 0])

In [9]:
# Define Pipeline = preprocessing + model

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter =500))
])

In [10]:
#Define CS Strategy
cv = StratifiedKFold(n_splits=10, shuffle = True, random_state =42)

In [11]:
# Define Scoring Metrics

scoring = {
    "accuracy" : "accuracy",
    "precision":"precision",
    "recall" : "recall",
    "f1":"f1",
    "roc_auc" : "roc_auc"
}

In [12]:
# Run Cross Validation
cv_results = cross_validate(
    pipe, X, y, cv =cv, scoring = scoring, return_train_score = False
)

In [13]:
for metric in scoring.keys():
    values = cv_results[f"test_{metric}"]
    print(f"{metric:>9}: {values.mean():.3f} +-{values.std():.3f}")

 accuracy: 0.975 +-0.020
precision: 0.974 +-0.028
   recall: 0.989 +-0.018
       f1: 0.981 +-0.015
  roc_auc: 0.995 +-0.007


## Observation

* Our 10-fold cross-validation results show very strong and stable performance.

* Accuracy is about 97.5% ± 2%, meaning across folds the model is consistently correct on most patients.

* Precision is ~97.4% — so when the model predicts cancer, it's right about 97% of the time.

* Recall is even higher at ~98.9% — the model successfully detects almost all actual cancer cases, which is critical in healthcare.

* The F1-score (98.1%) balances precision and recall, confirming overall robustness.

* Finally, ROC-AUC is ~0.995 — nearly perfect separation between patients with and without cancer.

The low standard deviations (small ± values) show that the model performs consistently across all folds, not just on one lucky fold.